# 11 · Would a single is-promo column help?

Promo posts get 1.5-1.9 times the views at equal likes and comments (notebook 09). Does one yes/no column improve the estimate, per platform? Paired on the same brand-grouped folds.

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "reach_samples_students.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
import eda_utils as eu, extra_models as em, modeling as mo, label_ablation as la

FIG = ROOT / "figures"; plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 220, "display.max_columns", 30, "display.max_colwidth", 80)

In [2]:
f, meta = em.load(ROOT); _, lab = la.label_frames(ROOT, f.index); R = {p: la.promo_flag(f, meta, lab, p) for p in mo.PLATS}

In [3]:
T = pd.concat([R[p][0] for p in mo.PLATS], ignore_index=True); T.round(3)

,platform,columns,model,without,with is_promo,gain,low,high
0,TikTok,likes + comments,ridge,0.299,0.291,0.008,0.005,0.011
1,TikTok,likes + comments,boosting (sklearn),0.306,0.298,0.007,0.004,0.011
2,TikTok,likes + comments,extra trees,0.295,0.289,0.005,0.002,0.009
3,TikTok,followers + reactions + flags,ridge,0.278,0.274,0.004,0.002,0.007
4,TikTok,followers + reactions + flags,boosting (sklearn),0.292,0.288,0.004,0.002,0.007
5,TikTok,followers + reactions + flags,extra trees,0.277,0.275,0.002,-0.000,0.005
6,YouTube,likes + comments,ridge,0.618,0.606,0.012,0.005,0.018
7,YouTube,likes + comments,boosting (sklearn),0.581,0.576,0.005,-0.000,0.011
8,YouTube,likes + comments,extra trees,0.576,0.568,0.008,0.002,0.013
9,YouTube,followers + reactions + flags,ridge,0.545,0.542,0.002,0.000,0.004


In [4]:
A = pd.DataFrame([R[p][1] for p in mo.PLATS]); A.round(2)

,platform,share promo,"median followers, not promo","median followers, promo"
0,TikTok,0.34,1356.5,1855.0
1,YouTube,0.42,25100.0,39400.0
2,X (Twitter),0.19,373.0,1524.0


## YouTube: where the large errors come from
Hypothesis: underestimated posts are promo. Compared with the state of the reaction columns.

In [5]:
yt = la.youtube_error_profile(f, meta, lab, eu.load(ROOT).astype({"mention_id": str})); display(yt["by_state"].round(2)); display(yt["by_error"].round(2)); yt["zero_posts"]

,state,n,MAE,bias,promo,median_views,share of error
0,has reactions,1769,0.38,0.00,0.4,590.0,0.60
1,likes hidden,106,0.55,0.01,0.5,9.0,0.05
2,zero likes and comments,566,0.68,0.00,0.5,231.0,0.35


,error,n,promo,zero
0,overestimated over x3,409,0.39,0.40
1,within x3,1630,0.43,0.15
2,underestimated over x3,402,0.46,0.39


{'n': 566,
 'share 1k+ views': 0.2862190812720848,
 'n 10k+ views': 30,
 'max views': 407071.9999999999}

## What if posts with zero likes and comments are removed?
Two effects kept apart: an easier test set, and cleaner training. Removing them from the test lowers the error (YouTube 0.46 to 0.38), but the same model is simply scored on easier posts. Removing them from training gains at most 0.002. So report both numbers rather than dropping them.

In [6]:
m_raw = eu.load(ROOT).astype({'mention_id': str}); Z = pd.DataFrame([la.drop_zero_reactions(f, meta, m_raw, p) for p in mo.PLATS]); json.dump(Z.to_dict('records'), open(ROOT / 'data' / 'processed' / 'zero_numbers.json', 'w'), indent=1, default=float); Z.round(3).T

,0,1,2
platform,TikTok,YouTube,X (Twitter)
posts,2491,2441,1803
zero share,0.036,0.272,0.55
"all posts, trained on all",0.277,0.455,0.481
zero posts only,0.534,0.655,0.51
"kept posts, trained on all",0.268,0.38,0.446
"kept posts, trained on kept",0.269,0.38,0.445
training gain,-0.001,0.0,0.002
low,-0.002,-0.003,-0.001
high,-0.0,0.004,0.004


## Takeaways
- **One is-promo column gives a real but very small gain:** at most about +0.006 with followers + reactions + flags (YouTube, boosting), up to +0.012 with likes + comments alone. Nothing on X.
- **Why the 1.5-1.9 factor does not become a gain:** that factor comes from a line on likes and comments only, and promo authors have more followers (median 39,400 against 25,100 on YouTube), so followers already carry much of it.
- **Promo does not explain YouTube's large errors:** underestimated posts are 46% promo against 43% of well-estimated ones, and 39% of overestimated.
- **Zero likes and comments do:** 23% of YouTube posts hold 35% of the error (MAE 0.68 against 0.38). 29% of them have at least 1,000 views and 30 have at least 10,000, up to about 400,000. Some of these zeros may be missing values stored as zero. This is a question for Brand24, not a finding.

In [7]:
json.dump({"gains": T.to_dict("records"), "authors": A.to_dict("records"), "youtube": {k: (v.to_dict("records") if hasattr(v, "to_dict") else v) for k, v in yt.items()}}, open(ROOT / "data" / "processed" / "promo_numbers.json", "w"), indent=1, default=float)